In [1]:
import psycopg2
import pandas as pd

In [2]:
# Connect to petadex
DB = {
    "host":     "petadex.ccz9y6yshbls.us-east-1.rds.amazonaws.com",
    "port":     5432,
    "database": "petadex",
    "user":     "readonly_user",
    "password": "petadex",
}

conn = psycopg2.connect(**DB)
cur  = conn.cursor()

In [3]:
cur.execute("""
    SELECT
        s.acc AS library_id,
        s.latitude,
        s.longitude,
        c.sequence_count
    FROM sra_metadata s
    JOIN (
        SELECT library_id, COUNT(*) AS sequence_count
        FROM logan_catalytic_orfs
        GROUP BY library_id
    ) c ON c.library_id = s.acc
    WHERE s.latitude IS NOT NULL AND s.longitude IS NOT NULL
""")

rows = cur.fetchall()
df = pd.DataFrame(rows, columns=["library_id", "latitude", "longitude", "sequence_count"])
df.to_parquet("petadex_library_geo_density.parquet", index=False)

In [3]:
# connect to Logan

DB = {
    "host":     "serratus-aurora-20210406.cluster-ro-ccz9y6yshbls.us-east-1.rds.amazonaws.com",
    "port":     5432,
    "database": "logan",
    "user":     "public_reader",
    "password": "serratus",
}

conn = psycopg2.connect(**DB)
cur  = conn.cursor()

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

server_cur = conn.cursor(name="biosample_geo_cursor")  # server-side cursor: streams from Postgres instead of buffering it all client-side
server_cur.itersize = 100_000

server_cur.execute("""
SELECT ST_X(lat_lon::geometry) AS lon, ST_Y(lat_lon::geometry) AS lat
FROM public.biosample_geographical_location;
""")

out_path = "logan_biosample_geo.parquet"
writer = None
total = 0

while True:
    batch = server_cur.fetchmany(100_000)
    if not batch:
        break
    df = pd.DataFrame(batch, columns=["lon", "lat"])
    table = pa.Table.from_pandas(df, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(out_path, table.schema)
    writer.write_table(table)
    total += len(batch)
    print(f"wrote {total} rows so far")

if writer is not None:
    writer.close()
server_cur.close()

wrote 100000 rows so far
wrote 200000 rows so far
wrote 300000 rows so far
wrote 400000 rows so far
wrote 500000 rows so far
wrote 600000 rows so far
wrote 700000 rows so far
wrote 800000 rows so far
wrote 900000 rows so far
wrote 1000000 rows so far
wrote 1100000 rows so far
wrote 1200000 rows so far
wrote 1300000 rows so far
wrote 1400000 rows so far
wrote 1500000 rows so far
wrote 1600000 rows so far
wrote 1700000 rows so far
wrote 1800000 rows so far
wrote 1900000 rows so far
wrote 2000000 rows so far
wrote 2100000 rows so far
wrote 2200000 rows so far
wrote 2300000 rows so far
wrote 2400000 rows so far
wrote 2500000 rows so far
wrote 2600000 rows so far
wrote 2700000 rows so far
wrote 2800000 rows so far
wrote 2900000 rows so far
wrote 3000000 rows so far
wrote 3100000 rows so far
wrote 3200000 rows so far
wrote 3300000 rows so far
wrote 3400000 rows so far
wrote 3500000 rows so far
wrote 3600000 rows so far
wrote 3700000 rows so far
wrote 3800000 rows so far
wrote 3900000 rows so

In [2]:
df = pd.read_parquet("logan_biosample_geo.parquet")

In [4]:
df.head(1)["lat_lon"].values[0]

'0101000020E610000055E0BFD4AAA762C057A30D76792120C0'

In [ ]:
# convert lat_lon to separate columns
df[["latitude", "longitude"]] = df["lat_lon"].str.split(",", expand=True).astype(float)